In [25]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from prophet import Prophet
import pmdarima as pm
# 폰트 설정 (한글 깨짐 방지, Windows 기준 'Malgun Gothic', Mac은 'AppleGothic')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [27]:
def calculate_metrics(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    # SMAPE: 백분율 오차 (0~100%), 작을수록 좋음
    smape = 100 * np.mean(2 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred)))
    
    print(f"\n[{model_name} 성능 평가]")
    print(f"RMSE  : {rmse:.4f}")
    print(f"MAE   : {mae:.4f}")
    print(f"SMAPE : {smape:.4f}%")
    return rmse, mae, smape

In [31]:
import pandas as pd
import numpy as np
import pmdarima as pm
from pmdarima.arima import auto_arima

# ==========================================
# 1. 데이터 로드 및 전처리
# ==========================================
print("[Auto-ARIMA] 데이터 준비 중...")
df = pd.read_csv('D:/2025-2/BDA/공모전/sample data/rtu_data_1hour_v2.csv')
df['datetime'] = pd.to_datetime(df['datetime'])

# 계수 계산
valid_data = df[df['activePow'] > 0]
BILL_RATE = (valid_data['electricityCost_won'] / valid_data['activePow']).mean()
CARBON_RATE = (valid_data['carbon_kgCO2'] / valid_data['activePow']).mean()

# 전체 합계 데이터 생성 (시간순 정렬 필수)
df_agg = df.groupby('datetime')['activePow'].sum().reset_index()
df_agg = df_agg.sort_values('datetime')
y_train = df_agg['activePow'].values

# ==========================================
# 2. Auto-ARIMA 모델 학습
# ==========================================
print("[Auto-ARIMA] 최적의 모델 탐색 및 학습 중 (시간이 소요됩니다)...")
model = pm.auto_arima(y_train, 
                      seasonal=True, 
                      m=24,             # 하루 주기 (24시간)
                      stepwise=True,    # 빠른 탐색
                      suppress_warnings=True)

print(f" -> 찾은 최적 모델: {model.summary()}")

# ==========================================
# 3. 5월 데이터 예측
# ==========================================
# 5월 1일 ~ 31일 (총 31일 * 24시간 = 744시간)
n_periods = 24 * 31
forecast, conf_int = model.predict(n_periods=n_periods, return_conf_int=True)

# 예측 날짜 생성
future_dates = pd.date_range(start='2025-05-01 00:00:00', periods=n_periods, freq='H')

# ==========================================
# 4. 결과 정리 및 파일 저장
# ==========================================
submission = pd.DataFrame({'id': future_dates, 'activePow': forecast})

# 음수 보정
submission['activePow'] = submission['activePow'].apply(lambda x: max(0, x))

# 누적값 계산
submission['accumActiveEnergy'] = submission['activePow'].cumsum()
submission['may_bill'] = (submission['activePow'] * BILL_RATE).cumsum()
submission['may_carbon'] = (submission['activePow'] * CARBON_RATE).cumsum()

# 파일 저장
save_path = 'D:/2025-2/BDA/공모전/sample data/result/final_result_arima.csv'
final_cols = ['id', 'activePow', 'accumActiveEnergy', 'may_bill', 'may_carbon']
submission[final_columns].to_csv(save_path, index=False)

print(f"[Auto-ARIMA] 완료! 파일 저장됨: {save_path}")
print(submission.head())

calculate_metrics(df_agg['activePow'].values, y_pred_arima, "Auto-ARIMA")

[Auto-ARIMA] 데이터 준비 중...
[Auto-ARIMA] 최적의 모델 탐색 및 학습 중 (시간이 소요됩니다)...


D:\Miniconda3\envs\bda\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
D:\Miniconda3\envs\bda\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
D:\Miniconda3\envs\bda\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
D:\Miniconda3\envs\bda\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
D:\Miniconda3\envs\bda\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
D:\Miniconda3\envs\b

 -> 찾은 최적 모델:                                  SARIMAX Results                                  
Dep. Variable:                          y   No. Observations:                 3601
Model:             SARIMAX(0, 0, [1], 24)   Log Likelihood              -21535.266
Date:                    Fri, 02 Jan 2026   AIC                          43076.531
Time:                            17:52:07   BIC                          43095.098
Sample:                                 0   HQIC                         43083.148
                                   - 3601                                         
Covariance Type:                      opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
intercept   3.913e+04      1.638   2.39e+04      0.000    3.91e+04    3.91e+04
ma.S.L24       0.0265      0.017      1.584      0.113      -0.006       0.059
sigma2

D:\Miniconda3\envs\bda\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
D:\Miniconda3\envs\bda\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


NameError: name 'y_pred_arima' is not defined

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet

# ==========================================
# 1. 데이터 로드 및 전처리
# ==========================================
print("[Prophet] 데이터 준비 중...")
# 경로가 맞는지 꼭 확인하세요
df = pd.read_csv('D:/2025-2/BDA/공모전/sample data/rtu_data_1hour_v2.csv')
df['datetime'] = pd.to_datetime(df['datetime'])

# 요금 및 탄소 배출 계수 계산
valid_data = df[df['activePow'] > 0]
BILL_RATE = (valid_data['electricityCost_won'] / valid_data['activePow']).mean()
CARBON_RATE = (valid_data['carbon_kgCO2'] / valid_data['activePow']).mean()

# Prophet용 데이터셋 생성 (시간대별 합계)
df_agg = df.groupby('datetime')['activePow'].sum().reset_index()
df_prophet = df_agg.rename(columns={'datetime': 'ds', 'activePow': 'y'})

# ==========================================
# 2. Prophet 모델 학습
# ==========================================
model = Prophet(yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=True)
model.add_country_holidays(country_name='KR')

print("[Prophet] 모델 학습 시작...")
model.fit(df_prophet)

# ==========================================
# 3. 5월 데이터 예측
# ==========================================
# 2025년 5월 28일 23시까지 예측하기 위해 기간 설정 (넉넉하게 잡고 자름)
future = model.make_future_dataframe(periods=24*40, freq='H') 

print("[Prophet] 미래 데이터 예측 중...")
# 여기서 forecast는 여러 컬럼(yhat, yhat_lower 등)을 가진 표입니다.
forecast = model.predict(future)

# 5월 1일 ~ 5월 28일 데이터만 추출
target_mask = (forecast['ds'] >= '2025-05-01 00:00:00') & (forecast['ds'] <= '2025-05-28 23:00:00')
forecast_may = forecast.loc[target_mask].copy()

# 음수 예측값 0으로 보정 (yhat 컬럼이 예측값입니다)
forecast_may['yhat'] = forecast_may['yhat'].apply(lambda x: max(0, x))

# ==========================================
# 4. 결과 정리 및 파일 저장
# ==========================================
submission = pd.DataFrame()
submission['id'] = forecast_may['ds']

# [수정된 부분] forecast 전체가 아니라 'yhat' 컬럼만 가져와야 함
submission['activePow'] = forecast_may['yhat']

# 누적값 및 요금 계산
submission['accumActiveEnergy'] = submission['activePow'].cumsum()
submission['may_bill'] = (submission['activePow'] * BILL_RATE).cumsum()
submission['may_carbon'] = (submission['activePow'] * CARBON_RATE).cumsum()

# 파일 저장
save_path = 'D:/2025-2/BDA/공모전/sample data/result/final_result_prophet.csv'
final_cols = ['id', 'activePow', 'accumActiveEnergy', 'may_bill', 'may_carbon']

submission[final_cols].to_csv(save_path, index=False)

print(f"[Prophet] 완료! 파일 저장됨: {save_path}")
print(submission.head())

calculate_metrics(df_prophet['activePow'].values, y_pred_prophet, "Prophet")